# Step 5: Verify & Troubleshoot

![Azure CLI](https://img.shields.io/badge/Azure_CLI-0078D4?logo=microsoftazure&logoColor=white)
![GitHub CLI](https://img.shields.io/badge/GitHub_CLI-181717?logo=github&logoColor=white)
![Security](https://img.shields.io/badge/Security-Best_Practices-critical?logo=shieldsdotio&logoColor=white)

This notebook verifies your entire setup and helps troubleshoot common issues.

**What this notebook does:**
1. Lists all federated credentials per identity
2. Verifies GitHub environments exist in each repo
3. Verifies GitHub secrets are configured
4. Provides common troubleshooting steps

## Load Configuration

In [ ]:
import subprocess, json, sys, os
from pathlib import Path

def run_cmd(cmd, capture=True, check=True):
    """Run a shell command and return the result."""
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and result.returncode != 0:
        print(f"ERROR: {result.stderr.strip()}")
        return None
    return result

# Load config
config_path = Path.cwd() / "config.json"
config = {}
if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)
    print(f"✓ Loaded config from {config_path}")
    # Validate required keys
    required_keys = ["org_name", "environments", "identity_type"]
    missing = [k for k in required_keys if k not in config or not config[k]]
    if missing:
        print(f"⚠ config.json is missing required keys: {missing}")
else:
    print(f"No config.json found at {config_path}")
    print("Tip: Copy config.json.example to config.json and fill in your values, or run notebook 01 first.")

GITHUB_OWNER = config.get("org_name") or input("GitHub owner (org name or personal username): ").strip()
ENVIRONMENTS = config.get("environments", ["dev", "staging", "production"])
IDENTITY_TYPE = config.get("identity_type") or input("Identity type ('sp' or 'mi'): ").strip().lower()
IDENTITIES = config.get("identities", {})
MI_RESOURCE_GROUP = config.get("mi_resource_group", "")
REPOS = config.get("repos", [])

# Detect account type
IS_ORG = False
r = run_cmd(f'gh api /orgs/{GITHUB_OWNER} --jq .login 2>/dev/null', check=False)
if r and r.returncode == 0 and r.stdout.strip():
    IS_ORG = True

print(f"GitHub owner: {GITHUB_OWNER} ({'organization' if IS_ORG else 'personal account'})")
print(f"Identity type: {'Service Principal' if IDENTITY_TYPE == 'sp' else 'Managed Identity'}")
print(f"Repositories: {len(REPOS)}")

## 1. Verify Federated Credentials

Lists all federated credentials for each identity to confirm they were created.

In [ ]:
# --- List Federated Credentials ---

for env in ENVIRONMENTS:
    print(f"\n{'='*50}")
    print(f"Environment: {env}")
    print(f"{'='*50}")

    if IDENTITY_TYPE == "mi":
        mi_name = IDENTITIES.get(env, {}).get("name", f"github-actions-{env}")
        cmd = (
            f'az identity federated-credential list '
            f'--identity-name "{mi_name}" '
            f'--resource-group "{MI_RESOURCE_GROUP}" '
            f'--query "[].{{name:name, subject:subject}}" -o table'
        )
    else:
        obj_id = IDENTITIES.get(env, {}).get("object_id", "")
        if not obj_id:
            obj_id = input(f"  SP Object ID for '{env}': ").strip()
        cmd = (
            f'az ad app federated-credential list '
            f'--id "{obj_id}" '
            f'--query "[].{{name:name, subject:subject}}" -o table'
        )

    r = run_cmd(cmd, check=False)
    if r and r.returncode == 0:
        print(r.stdout)
    else:
        print(f"  ✗ Failed to list credentials: {r.stderr.strip() if r else ''}")

## 2. Verify GitHub Environments

Checks that each repository has the expected environments.

In [ ]:
# --- Verify GitHub Environments ---

if not REPOS:
    repos_input = input("Enter repo names to check (comma-separated): ").strip()
    REPOS = [r.strip() for r in repos_input.split(",") if r.strip()]

env_results = []

for repo in REPOS:
    cmd = (
        f'gh api "/repos/{GITHUB_OWNER}/{repo}/environments" '
        f'--jq ".environments[].name" 2>/dev/null'
    )
    r = run_cmd(cmd, check=False)

    if r and r.returncode == 0:
        existing_envs = set(r.stdout.strip().split("\n")) if r.stdout.strip() else set()
        for env in ENVIRONMENTS:
            found = env in existing_envs
            env_results.append({"repo": repo, "env": env, "status": "✓" if found else "✗ Missing"})
    else:
        for env in ENVIRONMENTS:
            env_results.append({"repo": repo, "env": env, "status": "? Error"})

header = f"{'Repo':<30} {'Environment':<15} {'Status':<15}"
print(header)
print("-" * len(header))
for r in env_results:
    print(f"{r['repo']:<30} {r['env']:<15} {r['status']:<15}")

## 3. Verify GitHub Secrets

Checks that the required secrets exist — at the organization level (org accounts) or repository level (personal accounts).

In [ ]:
# --- Verify GitHub Secrets ---

env_secret_map = {"dev": "DEV", "staging": "STAGING", "production": "PROD"}
expected_secrets = ["AZURE_TENANT_ID", "AZURE_SUBSCRIPTION_ID"]
for env in ENVIRONMENTS:
    suffix = env_secret_map.get(env, env.upper())
    expected_secrets.append(f"AZURE_CLIENT_ID_{suffix}")

if IS_ORG:
    # Organization — check org-level secrets
    r = run_cmd(f'gh secret list --org "{GITHUB_OWNER}" 2>&1', check=False)
    if r and r.returncode == 0:
        existing_secrets = r.stdout
        print("Organization secrets found:\n")
        for secret in expected_secrets:
            found = secret in existing_secrets
            status = "✓" if found else "✗ Missing"
            print(f"  {status}  {secret}")
    else:
        print(f"✗ Could not list org secrets: {r.stderr.strip() if r else ''}")
        print("  You may not have admin access to the organization.")
else:
    # Personal account — check repo-level secrets
    print("Checking repository-level secrets (personal account):\n")
    for repo in REPOS:
        print(f"  {repo}:")
        r = run_cmd(f'gh secret list --repo "{GITHUB_OWNER}/{repo}" 2>&1', check=False)
        if r and r.returncode == 0:
            existing_secrets = r.stdout
            for secret in expected_secrets:
                found = secret in existing_secrets
                status = "✓" if found else "✗ Missing"
                print(f"    {status}  {secret}")
        else:
            print(f"    ✗ Could not list secrets: {r.stderr.strip() if r else ''}")

## 4. Verify Azure Authentication

Checks that Azure CLI and GitHub CLI are properly configured.

In [ ]:
# --- Verify Azure CLI ---

print("Azure CLI:")
r = run_cmd("az account show --output json", check=False)
if r and r.returncode == 0:
    acct = json.loads(r.stdout)
    print(f"  ✓ Logged in as: {acct.get('user', {}).get('name', 'unknown')}")
    print(f"  ✓ Subscription: {acct.get('name', 'unknown')}")
    print(f"  ✓ Tenant: {acct.get('tenantId', 'unknown')}")
else:
    print("  ✗ Not authenticated. Run: az login")

print("\nGitHub CLI:")
r = run_cmd("gh auth status 2>&1", check=False)
if r and r.returncode == 0:
    # gh auth status outputs to stderr
    output = r.stdout + r.stderr
    print(f"  ✓ Authenticated")
    for line in output.strip().split("\n"):
        if "Logged in" in line or "account" in line.lower():
            print(f"    {line.strip()}")
else:
    print("  ✗ Not authenticated. Run: gh auth login")

## Common Issues & Fixes

### Credential Already Exists
If a credential already exists with the same name, the setup script skips it. This is safe — no action needed.

### Permission Errors
Ensure you have:
- **Service Principal:** `Application.ReadWrite.All` or Owner role on the SP
- **Managed Identity:** `Contributor` on the MI resource group

### OIDC Login Fails in GitHub Actions
Common causes:
1. **Wrong client ID** — check that the secret value matches the identity's `appId` (SP) or `clientId` (MI)
2. **Wrong subject claim** — the federated credential subject must exactly match `repo:<org>/<repo>:environment:<env>`
3. **Environment not set** — the workflow must use `environment:` in the job definition
4. **Missing `id-token: write` permission** — required in the workflow's `permissions` block

### Verify a Specific Credential

In [ ]:
# --- Check a specific credential ---
# Enter a repo name and environment to verify the credential exists

check_repo = input("Repo to check (blank to skip): ").strip()
if check_repo:
    check_env = input("Environment (dev/staging/production): ").strip()
    cred_name = f"{check_repo}-{check_env}"
    expected_subject = f"repo:{GITHUB_OWNER}/{check_repo}:environment:{check_env}"

    print(f"\nLooking for credential: {cred_name}")
    print(f"Expected subject: {expected_subject}")

    if IDENTITY_TYPE == "mi":
        mi_name = IDENTITIES.get(check_env, {}).get("name", f"github-actions-{check_env}")
        cmd = (
            f'az identity federated-credential show '
            f'--identity-name "{mi_name}" '
            f'--resource-group "{MI_RESOURCE_GROUP}" '
            f'--name "{cred_name}" --output json'
        )
    else:
        obj_id = IDENTITIES.get(check_env, {}).get("object_id", "")
        if not obj_id:
            obj_id = input(f"  SP Object ID for '{check_env}': ").strip()
        cmd = (
            f'az ad app federated-credential show '
            f'--id "{obj_id}" '
            f'--federated-credential-id "{cred_name}" --output json'
        )

    r = run_cmd(cmd, check=False)
    if r and r.returncode == 0:
        cred = json.loads(r.stdout)
        actual_subject = cred.get("subject", "")
        match = actual_subject == expected_subject
        print(f"\n  ✓ Credential found: {cred_name}")
        print(f"  Subject: {actual_subject}")
        print(f"  Subject match: {'✓ Yes' if match else '✗ NO — mismatch!'}")
        print(f"  Issuer: {cred.get('issuer', '')}")
        print(f"  Audiences: {cred.get('audiences', [])}")
    else:
        print(f"\n  ✗ Credential not found: {cred_name}")
        print(f"    {r.stderr.strip()[:200] if r else ''}")
else:
    print("Skipped.")

## Delete a Credential (Cleanup)

Use this when decommissioning a repository.

In [ ]:
# --- Delete a credential ---

del_repo = input("Repo to remove credentials for (blank to skip): ").strip()
if del_repo:
    confirm = input(f"Delete ALL federated credentials for '{del_repo}'? (yes/no): ").strip().lower()
    if confirm == "yes":
        for env in ENVIRONMENTS:
            cred_name = f"{del_repo}-{env}"
            print(f"Deleting: {cred_name}")

            if IDENTITY_TYPE == "mi":
                mi_name = IDENTITIES.get(env, {}).get("name", f"github-actions-{env}")
                cmd = (
                    f'az identity federated-credential delete '
                    f'--identity-name "{mi_name}" '
                    f'--resource-group "{MI_RESOURCE_GROUP}" '
                    f'--name "{cred_name}" --yes'
                )
            else:
                obj_id = IDENTITIES.get(env, {}).get("object_id", "")
                if not obj_id:
                    obj_id = input(f"  SP Object ID for '{env}': ").strip()
                cmd = (
                    f'az ad app federated-credential delete '
                    f'--id "{obj_id}" '
                    f'--federated-credential-id "{cred_name}"'
                )

            r = run_cmd(cmd, check=False)
            if r and r.returncode == 0:
                print(f"  ✓ Deleted: {cred_name}")
            else:
                print(f"  ✗ Failed: {r.stderr.strip()[:200] if r else ''}")
    else:
        print("Cancelled.")
else:
    print("Skipped.")

## Summary

If all checks pass above, your setup is complete! Your GitHub Actions workflows can now authenticate to Azure using OIDC — no stored secrets needed.

**To test:** Go to any configured repo → **Actions** → **Deploy to Azure** → **Run workflow** → select an environment.

For more details, see the [FAQ & Troubleshooting](../FAQ.md) guide.